# Task 1: LSTM Autoencoder for Multi-Genre Music

> **Problem statement**
- Build an unsupervised sequence model that learns compact representations of symbolic music from the preprocessed dataset.
- Use the token/event sequences created in preprocessing (`data/processed/{train,val,test}` chunks).
- Train an LSTM autoencoder to reconstruct input token sequences.
- Evaluate reconstruction quality on validation/test splits and save the trained checkpoint for later tasks.

> **What this notebook does**
1. Loads chunked preprocessing outputs and builds train/val/test datasets.
2. Converts variable-length token sequences to padded fixed-length tensors.
3. Trains an LSTM encoder-decoder autoencoder with masked reconstruction loss.
4. Reports reconstruction metrics and qualitative reconstruction examples.
5. Saves model artifacts for downstream use.

In [1]:
from __future__ import annotations

import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

In [2]:
# Why: keep reusable data/model utilities in one place for clean, repeatable runs.

@dataclass
class Task1Config:
    project_root: Path = Path("..").resolve()
    processed_dir: Path = project_root / "data" / "processed"
    metadata_file: Path = processed_dir / "preprocessing_metadata.json"
    artifacts_dir: Path = project_root / "artifacts" / "task1_lstm_ae"

    seq_len: int = 256
    batch_size: int = 16
    num_workers: int = 0

    embedding_dim: int = 128
    hidden_dim: int = 256
    num_layers: int = 2
    dropout: float = 0.2

    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 5
    grad_clip: float = 1.0

    # Set to None for full-dataset training.
    max_train_examples: int | None = 40000
    max_val_examples: int | None = 5000
    max_test_examples: int | None = 5000


def load_split_token_sequences(
    split: str,
    max_examples: int | None,
    manifest_map: dict,
    processed_dir: Path,
    vocab_size: int,
) -> List[List[int]]:
    manifest = manifest_map.get(split, [])
    sequences: List[List[int]] = []

    for rel in tqdm(manifest, desc=f"Loading {split}", unit="chunk"):
        chunk_path = processed_dir / rel
        if not chunk_path.exists():
            continue
        with np.load(chunk_path, allow_pickle=True) as data:
            raw = data["token_sequences"].tolist()
            for seq in raw:
                if not seq:
                    continue
                shifted = [min(int(t) + 1, vocab_size - 1) for t in seq]
                sequences.append(shifted)
                if max_examples is not None and len(sequences) >= max_examples:
                    return sequences
    return sequences


class TokenSequenceDataset(Dataset):
    def __init__(self, sequences: List[List[int]], seq_len: int, pad_id: int = 0):
        self.sequences = sequences
        self.seq_len = seq_len
        self.pad_id = pad_id

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, idx: int) -> torch.Tensor:
        seq = self.sequences[idx]
        x = np.full((self.seq_len,), self.pad_id, dtype=np.int64)
        n = min(len(seq), self.seq_len)
        x[:n] = np.array(seq[:n], dtype=np.int64)
        return torch.from_numpy(x)


class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_dim: int,
        num_layers: int,
        dropout: float,
        pad_id: int,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.encoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.decoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.output_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)
        _, (h, c) = self.encoder(emb)
        dec_out, _ = self.decoder(emb, (h, c))
        logits = self.output_head(dec_out)
        return logits


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    pad_id: int,
    device: torch.device,
    train: bool = False,
    optimizer: torch.optim.Optimizer | None = None,
    grad_clip: float = 1.0,
    desc: str = "eval",
) -> Tuple[float, float]:
    if train and optimizer is None:
        raise ValueError("optimizer is required when train=True")

    model.train(train)
    total_loss = 0.0
    total_tokens = 0
    total_correct = 0

    for x in tqdm(loader, desc=desc, leave=False):
        x = x.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits.reshape(-1, logits.size(-1)), x.reshape(-1))

        if train:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        with torch.no_grad():
            mask = x != pad_id
            preds = logits.argmax(dim=-1)
            total_correct += ((preds == x) & mask).sum().item()
            total_tokens += mask.sum().item()
            total_loss += loss.item() * mask.sum().item()

    mean_loss = total_loss / max(total_tokens, 1)
    token_acc = total_correct / max(total_tokens, 1)
    return mean_loss, token_acc


def trim_pad(tokens: List[int], pad: int = 0) -> List[int]:
    out = []
    for t in tokens:
        if t == pad:
            break
        out.append(t)
    return out

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

cfg = Task1Config()
cfg.artifacts_dir.mkdir(parents=True, exist_ok=True)
assert cfg.metadata_file.exists(), f"Missing metadata: {cfg.metadata_file}"
print("Using metadata:", cfg.metadata_file)

with open(cfg.metadata_file, "r", encoding="utf-8") as f:
    meta = json.load(f)

duration_bins = tuple(meta["config"]["duration_bins"])
velocity_bins = tuple(meta["config"]["velocity_bins"])

token_space = 128 * (len(duration_bins) + 1) * (len(velocity_bins) + 1)
pad_id = 0
vocab_size = token_space + 1
print("token_space:", token_space, "vocab_size:", vocab_size)

train_sequences = load_split_token_sequences(
    "train",
    cfg.max_train_examples,
    meta["chunk_manifests"],
    cfg.processed_dir,
    vocab_size,
    )
val_sequences = load_split_token_sequences(
    "val",
    cfg.max_val_examples,
    meta["chunk_manifests"],
    cfg.processed_dir,
    vocab_size,
    )
test_sequences = load_split_token_sequences(
    "test",
    cfg.max_test_examples,
    meta["chunk_manifests"],
    cfg.processed_dir,
    vocab_size,
    )

print({
    "train": len(train_sequences),
    "val": len(val_sequences),
    "test": len(test_sequences),
})

Device: cpu
Using metadata: G:\Unsupervised-Neural-Network-for-Multi-Genre-Music-Generation-CSE425-\data\processed\preprocessing_metadata.json
token_space: 8064 vocab_size: 8065


Loading train:   0%|          | 0/15 [00:00<?, ?chunk/s]

Loading val:   0%|          | 0/2 [00:00<?, ?chunk/s]

Loading test:   0%|          | 0/2 [00:00<?, ?chunk/s]

{'train': 3798, 'val': 435, 'test': 487}


In [4]:
train_ds = TokenSequenceDataset(train_sequences, cfg.seq_len, pad_id)
val_ds = TokenSequenceDataset(val_sequences, cfg.seq_len, pad_id)
test_ds = TokenSequenceDataset(test_sequences, cfg.seq_len, pad_id)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

print("Batches:", {
    "train": len(train_loader),
    "val": len(val_loader),
    "test": len(test_loader),
})

Batches: {'train': 238, 'val': 28, 'test': 31}


In [5]:
model = LSTMAutoencoder(
    vocab_size=vocab_size,
    embedding_dim=cfg.embedding_dim,
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    dropout=cfg.dropout,
    pad_id=pad_id,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
    betas=(0.9, 0.999),
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

best_val_loss = float("inf")
history = []

for epoch in range(1, cfg.epochs + 1):
    train_loss, train_acc = run_epoch(
        model,
        train_loader,
        criterion,
        pad_id,
        DEVICE,
        train=True,
        optimizer=optimizer,
        grad_clip=cfg.grad_clip,
        desc=f"train e{epoch}",
    )
    val_loss, val_acc = run_epoch(
        model,
        val_loader,
        criterion,
        pad_id,
        DEVICE,
        train=False,
        desc=f"val e{epoch}",
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
    }
    history.append(row)
    print(row)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        ckpt_path = cfg.artifacts_dir / "best_lstm_autoencoder.pt"
        torch.save(
            {
                "model_state": model.state_dict(),
                "config": cfg.__dict__,
                "vocab_size": vocab_size,
                "pad_id": pad_id,
                "history": history,
            },
            ckpt_path,
        )
        print(f"Saved best checkpoint -> {ckpt_path}")

Model parameters: 4,948,225


train e1:   0%|          | 0/238 [00:00<?, ?it/s]

val e1:   0%|          | 0/28 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 6.223009454608784, 'val_loss': 3.7787258034517435, 'train_acc': 0.1566342329136024, 'val_acc': 0.5431314236126804}
Saved best checkpoint -> G:\Unsupervised-Neural-Network-for-Multi-Genre-Music-Generation-CSE425-\artifacts\task1_lstm_ae\best_lstm_autoencoder.pt


train e2:   0%|          | 0/238 [00:00<?, ?it/s]

...

val e4:   0%|          | 0/28 [00:00<?, ?it/s]

{'epoch': 4, 'train_loss': 0.15825088017081948, 'val_loss': 0.09627414953107682, 'train_acc': 0.9914526835053159, 'val_acc': 0.9942147938568342}
Saved best checkpoint -> G:\Unsupervised-Neural-Network-for-Multi-Genre-Music-Generation-CSE425-\artifacts\task1_lstm_ae\best_lstm_autoencoder.pt


train e5:   0%|          | 0/238 [00:00<?, ?it/s]

val e5:   0%|          | 0/28 [00:00<?, ?it/s]

{'epoch': 5, 'train_loss': 0.07542080102792184, 'val_loss': 0.056796175939290046, 'train_acc': 0.9965689261861036, 'val_acc': 0.9965831126216927}
Saved best checkpoint -> G:\Unsupervised-Neural-Network-for-Multi-Genre-Music-Generation-CSE425-\artifacts\task1_lstm_ae\best_lstm_autoencoder.pt


In [7]:
# Load best checkpoint for final evaluation
ckpt_path = cfg.artifacts_dir / "best_lstm_autoencoder.pt"
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"
state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model_state"])
model.eval()

test_loss, test_acc = run_epoch(
    model,
    test_loader,
    criterion,
    pad_id,
    DEVICE,
    train=False,
    desc="test",
)
test_perplexity = math.exp(min(test_loss, 20.0))
print({
    "test_loss": test_loss,
    "test_token_accuracy": test_acc,
    "test_perplexity": test_perplexity,
})

x = next(iter(test_loader)).to(DEVICE)
with torch.no_grad():
    logits = model(x)
    preds = logits.argmax(dim=-1)

sample_idx = 0
true_tokens = x[sample_idx].cpu().tolist()
pred_tokens = preds[sample_idx].cpu().tolist()

true_trim = trim_pad(true_tokens, pad_id)
pred_trim = trim_pad(pred_tokens, pad_id)

print("Ground truth tokens (first 80):", true_trim[:80])
print("Reconstructed tokens (first 80):", pred_trim[:80])

with open(cfg.artifacts_dir / "training_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)
print("Saved history ->", cfg.artifacts_dir / "training_history.json")

# Export Task 1 deliverables in dedicated, reusable locations.
deliverables_dir = cfg.artifacts_dir / "deliverables"
plots_dir = deliverables_dir / "plots"
midi_dir = deliverables_dir / "midi"
for d in (plots_dir, midi_dir):
    d.mkdir(parents=True, exist_ok=True)

# 1) Save reconstruction loss curve plot.
import matplotlib.pyplot as plt

epochs = [row["epoch"] for row in history]
train_losses = [row["train_loss"] for row in history]
val_losses = [row["val_loss"] for row in history]

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, marker="o", label="Train Reconstruction Loss")
plt.plot(epochs, val_losses, marker="o", label="Val Reconstruction Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Reconstruction Loss")
plt.title("Task 1 Reconstruction Loss Curve")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

loss_curve_path = plots_dir / "reconstruction_loss_curve.png"
plt.savefig(loss_curve_path, dpi=160)
plt.close()
print("Saved reconstruction loss curve ->", loss_curve_path)

# 2) Save 5 reconstructed MIDI samples.
import pretty_midi

duration_bin_count = len(duration_bins) + 1
stride = 128 * duration_bin_count

def decode_token_to_note(token_shifted: int) -> tuple[int, float, int] | None:
    if token_shifted <= 0:
        return None
    token = int(token_shifted) - 1
    if token < 0 or token >= token_space:
        return None

    v_bin = token // stride
    rem = token % stride
    d_bin = rem // 128
    pitch = rem % 128

    if d_bin < len(duration_bins):
        duration = float(duration_bins[d_bin])
    else:
        duration = float(duration_bins[-1])

    if v_bin < len(velocity_bins):
        velocity = int(velocity_bins[v_bin])
    else:
        velocity = 127

    return pitch, duration, velocity

with torch.no_grad():
    sample_batch = next(iter(test_loader)).to(DEVICE)
    sample_logits = model(sample_batch)
    sample_preds = sample_logits.argmax(dim=-1).cpu().tolist()

num_to_save = min(5, len(sample_preds))
midi_paths = []
for i in range(num_to_save):
    pred_seq = trim_pad(sample_preds[i], pad_id)
    pm = pretty_midi.PrettyMIDI(initial_tempo=120.0)
    inst = pretty_midi.Instrument(program=0, name="reconstructed")

    t = 0.0
    for tok in pred_seq:
        decoded = decode_token_to_note(tok)
        if decoded is None:
            continue
        pitch, duration, velocity = decoded
        end_t = t + max(0.05, duration)
        inst.notes.append(
            pretty_midi.Note(
                velocity=max(1, min(127, int(velocity))),
                pitch=max(0, min(127, int(pitch))),
                start=float(t),
                end=float(end_t),
            )
        )
        t = end_t

    pm.instruments.append(inst)
    out_path = midi_dir / f"reconstruction_sample_{i + 1:02d}.mid"
    pm.write(str(out_path))
    midi_paths.append(str(out_path))

index_path = midi_dir / "midi_samples_index.json"
with open(index_path, "w", encoding="utf-8") as f:
    json.dump({"files": midi_paths}, f, indent=2)

print("Saved MIDI samples ->", midi_dir)
print("Saved MIDI index ->", index_path)

test:   0%|          | 0/31 [00:00<?, ?it/s]

{'test_loss': 0.04914274357415081, 'test_token_accuracy': 0.9979192274629839, 'test_perplexity': 1.05037027364655}
Ground truth tokens (first 80): [5541, 5416, 4530, 5428, 5684, 3771, 4669, 4672, 4547, 4675, 3655, 4553, 4556, 6439, 5425, 4785, 3771, 4669, 3776, 4547, 4675, 3653, 4551, 4553, 3660, 2757, 6437, 5423, 5679, 3771, 4669, 3776, 4547, 3779, 4548, 4551, 4553, 4556, 3652, 6439, 5546, 6315, 5675, 4537, 3771, 4669, 3776, 4547, 3779, 4551, 4553, 4556, 5433, 5541, 3771, 4669, 3776, 4547, 4675, 4551, 4553, 4556, 6439, 5413, 4520, 3771, 4669, 3776, 4547, 3779, 4551, 4553, 4556, 6437, 5416, 5544, 5419, 5547, 3771, 3773]
Reconstructed tokens (first 80): [5541, 5416, 4530, 5428, 5684, 3771, 4669, 4672, 4547, 4675, 3655, 4553, 4556, 6439, 5425, 4785, 3771, 4669, 3776, 4547, 4675, 3653, 4551, 4553, 3660, 2757, 6437, 5423, 5679, 3771, 4669, 3776, 4547, 3779, 4548, 4551, 4553, 4556, 3652, 6439, 5546, 6315, 5675, 4537, 3771, 4669, 3776, 4547, 3779, 4551, 4553, 4556, 5433, 5541, 3771, 4669, 37